## SIMULADOR


In [17]:
## IMPORTS
import  json
from typing import List, Dict, Set
from pathlib import Path

print('[*] Importando os pacotes')


[*] Importando os pacotes


In [18]:
class Automato:
    def __init__(self, automato: Dict):
        self.alfabeto = automato['Sigma']
        self.estado_inicial = automato['q0']
        self.estados_finais = automato['F']
        self.transicoes = automato['delta']
        self.tipo_automato = self.classificar_automato()

    def classificar_automato(self) -> str:

        for estado, trans in self.transicoes.items():
            for situacao, destinos in trans.items():
                if len(destinos) !=1:
                    return 'AFN'
        return 'AFD'


    def processar_palavra(self, palavra: str) -> bool:
        if(self.tipo_automato == 'AFD'):
            return self.__processar_afd__(palavra)
        else:
            return self.__processar_afn__(palavra)

    def __processar_afd__(self,palavra)-> bool:
        estado_atual  = self.estado_inicial
        print(f"[*] Testando a palavra no AFD {palavra}")
        print('[*] o estado inicial é: ', estado_atual)
        for caractere in palavra:
            if caractere not in self.alfabeto:
                return False

            transicoes_estado = self.transicoes.get(estado_atual, {})

            if caractere not in transicoes_estado:
                return False
            estado_atual = transicoes_estado[caractere][0]
            print(f"[*] Leu {caractere} e foi pro estado: {estado_atual}")
        print('[*] teste finalizado')
        resultado_teste= True if estado_atual in self.estados_finais else False
        # print(f'{palavra} pertence ao afd: {resultado_teste}')
        return resultado_teste

    def __processar_afn__(self, palavra: str) -> bool:
        estados_atuais: Set[str] = {self.estado_inicial}

        print(f"[*] Testando a palavra no AFN: '{palavra}'")
        print(f"[*] Os estados iniciais são: {estados_atuais}")

        for caractere in palavra:
            if caractere not in self.alfabeto:
                print(f"[-] Caractere '{caractere}' não pertence ao alfabeto. Palavra rejeitada.")
                return False

            proximos_estados: Set[str] = set()

            for estado in estados_atuais:
                transicoes_estado = self.transicoes.get(estado, {})
                if caractere in transicoes_estado:
                    proximos_estados.update(transicoes_estado[caractere])

            estados_atuais = proximos_estados
            print(f"[*] Leu '{caractere}' e ramificou para os estados: {estados_atuais}")

            if not estados_atuais:
                print("[-] Caminho sem saída encontrado (conjunto de estados vazio). Palavra rejeitada.")
                return False

        print('[*] Teste finalizado')
        resultado_teste = any(estado in self.estados_finais for estado in estados_atuais)

        return resultado_teste

    def rodarTestes(self, palavrasTestes: List[str]):
        print('[*] Testando palavras testes')
        for palavra in palavrasTestes:
            resultado = self.processar_palavra(palavra)
            print(f"[*] {palavra}: {resultado}")
            print('-----------------------------------------------------')


In [1]:
def carregar_automatos():
    path_data = Path('./data')
    automatos = []

    for arquivo in path_data.glob('*.json'):
        with open(arquivo) as json_file:
            try:
                dados = json.load(json_file)
                automatos.append((arquivo.name, dados))
                print(f"Lido com sucesso: {arquivo.name}")
            except json.JSONDecodeError:
                print(f"Erro de formatação no arquivo: {arquivo.name}")

    print("[*] Tudo lido")
    return automatos